# 04.01 · Middleware mental model and hooks

**Module 04 · Middleware, Guardrails & Human-in-the-Loop** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Module 02 (`create_agent`, tools) and Module 03 (checkpointers, `wrap_model_call` in 03.03/03.05). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** where the six middleware hooks run in the agent loop: `before_agent`, `before_model`, `wrap_model_call`, `wrap_tool_call`, `after_model`, `after_agent`.
2. **Predict** the execution order when several middleware are stacked (before: in order, after: reversed, wrap: nested).
3. **Choose** between node-style hooks (update state, `jump_to`) and wrap-style hooks (short-circuit, retry, change the request).
4. **Debug** a failing middleware with tracebacks, `%xmode`, `%debug`, `%pdb` and a reproducible failing cell (this module's Jupyter skill).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Middleware overview](https://docs.langchain.com/oss/python/langchain/middleware/overview) | **Main page.** What middleware is for, the loop diagram, middleware inside a LangGraph workflow |
| [Custom middleware → Hooks](https://docs.langchain.com/oss/python/langchain/middleware/custom#hooks) | The hook table, [execution order](https://docs.langchain.com/oss/python/langchain/middleware/custom#execution-order) and [agent jumps](https://docs.langchain.com/oss/python/langchain/middleware/custom#agent-jumps) |
| [What's new in LangChain v1](https://docs.langchain.com/oss/python/releases/langchain-v1) | Why middleware replaced `pre_model_hook` / `post_model_hook` of the old prebuilt agent |
| [IPython magics: `%debug`, `%pdb`, `%xmode`](https://ipython.readthedocs.io/en/stable/interactive/magics.html) | The debugging skill (Part 5) |
| [Python `pdb` commands](https://docs.python.org/3/library/pdb.html#debugger-commands) | `p`, `pp`, `up`, `down`, `l`, `q` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/04-middleware-guardrails-hitl/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence one known, harmless warning (same as Lesson 02.04).
# langchain-core 1.6.9 and 1.6.10 serialize the injected ToolRuntime/Runtime while parsing input and Pydantic
# warns "PydanticSerializationUnexpectedValue ... field_name='context'". Context still arrives intact.
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

**You should now see** your repo root, key status (`set` / `not set`, never the value) and the two model IDs. No errors.

---
## Part 1 · Where middleware sits

An agent built by `create_agent` is a loop: **call the model → if it asked for tools, run them → call the model again → … → stop when it answers without tool calls.** Middleware lets you run your own code at fixed points of that loop **without rewriting it**:

```text
invoke ──► before_agent ──► ┌─► before_model ──► wrap_model_call( model ) ──► after_model ─┐
                            │                                                              │ tool calls?
                            └──────────── wrap_tool_call( tool ) ◄──── yes ◄──────────────┘
                                                                          no ──► after_agent ──► result
```

| Hook | Runs | Style | Async twin | Typical use |
|---|---|---|---|---|
| `before_agent` | once per `invoke`, first | node | `abefore_agent` | authenticate, load context, block a request |
| `before_model` | before **every** model call | node | `abefore_model` | trim messages, check limits |
| `wrap_model_call` | **around** every model call | wrap | `awrap_model_call` | retry, fallback, change tools/prompt, cache |
| `after_model` | after every model reply | node | `aafter_model` | inspect tool calls, human approval, validate |
| `wrap_tool_call` | **around** every tool call | wrap | `awrap_tool_call` | retry, permission check, logging |
| `after_agent` | once per `invoke`, last | node | `aafter_agent` | final output check, save memories |

**Node-style** hooks are graph nodes: they receive the state and may return a **state update** (a dict). **Wrap-style** hooks receive the request plus a `handler` (the next layer, eventually the real model or tool) and decide whether, how and how often to call it.

> **Sync or async?** This module runs agents with `invoke`. Under `ainvoke`/`astream` (Module 05 and production servers), a sync **node-style** hook still works, but a sync-only **wrap-style** hook raises `NotImplementedError: Asynchronous implementation of awrap_model_call is not available` (and an async-only one fails under `invoke`). Reusable middleware that wraps calls defines **both** `wrap_model_call` and `awrap_model_call` (Lesson 04.03; Lesson 05.03 runs it async). The `mastery.middleware` classes already do.

---
## Part 2 · See every hook fire

A tracer middleware that only prints. It subclasses `AgentMiddleware` and implements all six hooks. Each prints one line, so the output is the execution order.

In [ ]:
# 🧪 Offline: a tracer with all six hooks, on a scripted agent that calls one tool.
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import AIMessage, HumanMessage
from langchain.tools import tool
from mastery.testing import ScriptedChatModel

LOG: list[str] = []


class Tracer(AgentMiddleware):
    """Prints (and records) every hook it passes through."""

    def __init__(self, label: str) -> None:
        super().__init__()
        self.label = label

    @property
    def name(self) -> str:          # Each middleware in a stack needs a unique name.
        return f"Tracer{self.label}"

    def _log(self, text: str) -> None:
        LOG.append(f"{self.label}.{text}")

    def before_agent(self, state, runtime) -> dict[str, Any] | None:
        self._log("before_agent")

    def before_model(self, state, runtime) -> dict[str, Any] | None:
        self._log(f"before_model   ({len(state['messages'])} messages)")

    def wrap_model_call(self, request, handler):
        self._log("wrap_model_call → in")
        response = handler(request)          # The next layer, eventually the model.
        self._log("wrap_model_call ← out")
        return response

    def after_model(self, state, runtime) -> dict[str, Any] | None:
        self._log("after_model")

    def wrap_tool_call(self, request, handler):
        self._log(f"wrap_tool_call → {request.tool_call['name']}")
        result = handler(request)            # The real tool.
        self._log("wrap_tool_call ←")
        return result

    def after_agent(self, state, runtime) -> dict[str, Any] | None:
        self._log("after_agent")


@tool
def get_time(city: str) -> str:
    """Return the local time in a city."""
    return f"It is 09:30 in {city}."


def scripted():
    return ScriptedChatModel(responses=[
        AIMessage("", tool_calls=[{"name": "get_time", "args": {"city": "Bogotá"}, "id": "call_1"}]),
        "It is 09:30 in Bogotá.",
    ])


agent = create_agent(scripted(), tools=[get_time], middleware=[Tracer("A")])
LOG.clear()
agent.invoke({"messages": [HumanMessage("What time is it in Bogotá?")]})
print("\n".join(f"{i:>2}. {line}" for i, line in enumerate(LOG, 1)))

**You should now see** 12 numbered lines: `A.before_agent`, then `before_model (1 messages)`, the model wrapped (`→ in`, `← out`), `after_model`, the tool wrapped (`→ get_time`, `←`), a **second** model round (`before_model (3 messages)` …), and finally `A.after_agent`.

### What just happened?

- `before_agent` and `after_agent` ran **once**; the model hooks ran **twice** (one per model call); the tool hook ran **once** (one tool call).
- Between the two model rounds the state grew from 1 to 3 messages: the AI's tool call and the `ToolMessage` were appended.
- The hooks returned `None`, which means "no state change". The agent behaved exactly as without middleware.

---
## Part 3 · Two middleware: who runs first?

Real agents stack several middleware (`middleware=[A, B, C]`). The order of the list matters. Predict the output **before** running the cell.

In [ ]:
# 🧪 Offline: the same agent with two tracers.
agent = create_agent(scripted(), tools=[get_time], middleware=[Tracer("A"), Tracer("B")])
LOG.clear()
agent.invoke({"messages": [HumanMessage("What time is it in Bogotá?")]})
first_round = LOG[: LOG.index("A.after_model") + 1]
print("First model round:")
print("\n".join(f"  {line}" for line in first_round))
print("Ending:", LOG[-2:])

**You should now see**

```text
First model round:
  A.before_agent
  B.before_agent
  A.before_model   (1 messages)
  B.before_model   (1 messages)
  A.wrap_model_call → in
  B.wrap_model_call → in
  B.wrap_model_call ← out
  A.wrap_model_call ← out
  B.after_model
  A.after_model
Ending: ['B.after_agent', 'A.after_agent']
```

### What just happened?

| Hook kind | Order for `[A, B]` | Rule |
|---|---|---|
| `before_*` | A, then B | first to last |
| `after_*` | B, then A | **last to first** |
| `wrap_*` | A( B( model ) ) | nested: the **first** middleware is the **outermost** layer |

Think of the list as layers of an onion around the model: A is the outer skin. That is why the docs say *"place critical middleware first"*: an outer retry or a guardrail sees everything the inner layers do. Lesson 04.02 uses this to decide where retry and fallback go.

---
## Part 4 · What hooks can **do**

Returning `None` changes nothing. The two styles have different powers:

| Power | Node-style (`before_*`, `after_*`) | Wrap-style (`wrap_*`) |
|---|---|---|
| Change state | return a dict, e.g. `{"messages": [...]}` | return `ExtendedModelResponse(command=Command(update=...))` (model) or a `Command` (tool) |
| Stop early | return `{"jump_to": "end"}` (needs `can_jump_to=["end"]`) | don't call `handler` and return your own response |
| Change the request | no | `handler(request.override(tools=..., messages=..., model=...))` |
| Call the model/tool several times | no | call `handler` in a loop (retry) |

Two examples. First a **node-style guard** that ends the run before the model is called when a conversation is too long:

In [ ]:
# 🧪 Offline: node-style: stop early with jump_to.
from langchain.agents.middleware import before_model, hook_config


@before_model(can_jump_to=["end"])
def max_messages(state, runtime) -> dict[str, Any] | None:
    if len(state["messages"]) > 2:
        return {"messages": [AIMessage("This conversation is too long; please start a new one.")],
                "jump_to": "end"}
    return None


model = scripted()
guarded = create_agent(model, tools=[get_time], middleware=[max_messages])
result = guarded.invoke({"messages": [HumanMessage("What time is it in Bogotá?")]})
print("model calls:", len(model.calls))
print("last message:", result["messages"][-1].content)

**You should now see** `model calls: 1` and the "too long" message. The first model call happened (1 message), the tool ran, and before the **second** call the state had 3 messages, so the guard jumped to the end. The scripted second answer was never used.

Second, a **wrap-style cache** that answers without calling the model at all (short-circuit):

In [ ]:
# 🧪 Offline: wrap-style: short-circuit with a cache.
from langchain.agents.middleware import wrap_model_call

CACHE = {"What are your opening hours?": "We are open 8:00–18:00, Monday to Friday."}


@wrap_model_call
def answer_from_cache(request, handler):
    question = request.messages[-1].content if request.messages else ""
    if question in CACHE:
        return AIMessage(CACHE[question])     # handler is never called: no model cost.
    return handler(request)


model = ScriptedChatModel(responses=["(the model answered)"])
cached = create_agent(model, tools=[], middleware=[answer_from_cache])
for q in ["What are your opening hours?", "Do you ship to Lima?"]:
    out = cached.invoke({"messages": [HumanMessage(q)]})
    print(f"{q:<30} → {out['messages'][-1].content}")
print("model calls:", len(model.calls))

**You should now see** the cached opening hours, `(the model answered)` for the Lima question, and `model calls: 1`.

### What just happened?

- `@before_model` and `@wrap_model_call` are **decorators** that turn one function into a middleware (Lesson 04.03 compares them with classes).
- `can_jump_to=["end"]` declares the jump up front: the agent graph needs to know which edges exist when it is built. Forgetting it is a classic **silent** bug: the jump is ignored and the model runs anyway (see Pitfalls).
- A wrap hook may return an `AIMessage` (or a `ModelResponse`) without calling `handler`. Lessons 04.02 (retry, fallback) and 04.03 (budget guard) use the other powers.

In [ ]:
# 💸 Live: the two-tracer agent with a real Groq model. Same hooks, same order.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live = create_agent(get_model("fast"), tools=[get_time], middleware=[Tracer("A"), Tracer("B")],
                        system_prompt="Use get_time for time questions. Answer in one sentence.")
    LOG.clear()
    out = live.invoke({"messages": [HumanMessage("What time is it in Bogotá?")]})
    print("\n".join(LOG))
    print("\nAnswer:", out["messages"][-1].content)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) the same pattern as Part 3. If the model answered **without** calling the tool, you see only one model round: middleware observes what the model decides, it doesn't change it (unless you make it).

---
## Part 5 · 🛠️ Jupyter skill: debugging middleware

Middleware code runs **inside** the agent graph, so a bug shows up as a long traceback from LangGraph internals. Four tools make it manageable.

**1. Find the hook in the traceback.** The last lines of a middleware error always name the node: `During task with name 'BuggyAudit.before_model' and id '…'`. That is *your* class and hook.

**2. A reproducible failing cell.** A notebook that crashes halfway breaks **Run All**. Wrap the bug behind a flag (`SHOW_BUG = False`): Run All stays green, and you flip the flag to study the failure.

In [ ]:
# 🧪 Offline: a buggy middleware: it expects a 'user_id' key that the state doesn't have.
import traceback


class BuggyAudit(AgentMiddleware):
    def before_model(self, state, runtime):
        print("user:", state["user_id"])     # BUG: KeyError, the state has no user_id.


buggy = create_agent(scripted(), tools=[get_time], middleware=[BuggyAudit()])

SHOW_BUG = False   # ← set to True, run this cell, then run the %debug cell below.
if SHOW_BUG:
    buggy.invoke({"messages": [HumanMessage("What time is it in Bogotá?")]})
else:
    try:
        buggy.invoke({"messages": [HumanMessage("What time is it in Bogotá?")]})
    except KeyError as error:
        last_lines = traceback.format_exc().strip().splitlines()[-3:]
        print("Caught:", repr(error))
        print("\n".join(last_lines))

**You should now see** `Caught: KeyError('user_id')` and three traceback lines, the last one `During task with name 'BuggyAudit.before_model' …`.

**3. Shorter tracebacks with `%xmode`.** `%xmode Minimal` shows only the error line; `Context` (default) shows code around each frame; `Verbose` adds every local variable. Run the next cell, flip `SHOW_BUG = True` above, re-run it, and compare. Then set it back.

In [ ]:
# 🧪 Offline: traceback modes. Minimal is great for "what failed?", Verbose for "why?".
%xmode Minimal

**You should now see** `Exception reporting mode: Minimal`. Switch back with `%xmode Context` when you need the full picture (the next cell does it for you).

**4. Post-mortem debugging with `%debug`** (try this by hand; it waits for your input, so it is not part of Run All):

1. Set `SHOW_BUG = True` in the buggy cell and run it. It fails.
2. In a **new** cell type `%debug` and run it. A `ipdb>` prompt opens **at the line that failed**, inside `BuggyAudit.before_model`.
3. Type `p list(state)` and Enter: the keys that *are* in the state (`['messages']`, so no `user_id`). Try `pp state["messages"]`, `l` (show code), `up` (go to the caller), `q` (quit).
4. Fix the bug (`state.get("user_id", "anonymous")`), set `SHOW_BUG = False` again.

`%pdb on` makes this automatic: **every** error opens the debugger. Useful while hunting, but turn it off (`%pdb off`) before Run All, or the notebook will wait for input forever.

**5. Minimal repro: call the hook directly.** A hook is a plain method. You can call it with a hand-made state, no agent, no model. This is how Lesson 04.03 unit-tests middleware.

In [ ]:
# 🧪 Offline: reproduce the bug in one line, then test the fix the same way.
%xmode Context
%pdb off

state = {"messages": [HumanMessage("hi")]}
try:
    BuggyAudit().before_model(state, runtime=None)
except KeyError as error:
    print("reproduced:", repr(error))


class FixedAudit(AgentMiddleware):
    def before_model(self, state, runtime):
        print("user:", state.get("user_id", "anonymous"))


FixedAudit().before_model(state, runtime=None)

**You should now see** `Exception reporting mode: Context`, `Automatic pdb calling has been turned OFF`, `reproduced: KeyError('user_id')` and `user: anonymous`.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `TypeError: ... before_model() missing 1 required positional argument` | Hook signature not `(self, state, runtime)` (class) or `(state, runtime)` (decorator) | Use exactly those parameter names: the agent passes `runtime` by name |
| `jump_to` is silently ignored: your message is added **and the model still runs** | No `can_jump_to=["end"]` on the hook | `@before_model(can_jump_to=["end"])`, or `@hook_config(can_jump_to=["end"])` on a class method |
| `AssertionError: Please remove duplicate middleware instances.` | Two instances of the same class (same `name`) in one stack | Give each a unique `name` (property) as in `Tracer`, or use one instance |
| Wrap hook "loses" the answer (`None` returned) | Forgot `return handler(request)` | Always return the handler's result (or your own response) |
| Run All hangs forever | `%pdb on` left on, or `%debug`/`input()` in a Run All cell | `%pdb off`; keep debugger cells out of Run All; restart the kernel to unblock |
| Traceback is 80 lines of LangGraph internals | Normal for graph code | Read the **last** lines (`During task with name ...`); `%xmode Minimal` |
| Changes to a middleware class don't apply | The agent was built with the old class | Rebuild the agent (re-run the cell that calls `create_agent`) |

## ⚠️ Production notes

> ⚠️ **Production note: order is a design decision; write it down.**
> `[guardrail, retry, fallback, audit]` and `[audit, fallback, retry, guardrail]` behave differently: who sees retries, who sees blocked requests, what gets logged. Keep the middleware list in one function (`build_agent()`), comment why each layer sits where it does, and cover the order with a test like Part 3.

> ⚠️ **Production note: middleware runs on every call, so keep it cheap.**
> A `before_model` that calls another model or a database doubles your latency per loop step. Measure (Lesson 04.02), cache, and move slow checks to `before_agent` (once per run) when they don't need per-step data.

## 🏋️ Exercises

1. **(Easy)** Build the agent with `[Tracer("B"), Tracer("A")]`. Write down the predicted order first, then run.
2. **(Medium)** Make `max_messages` allow **one** tool round but stop before a **third** model call. Which threshold do you need, and why?
3. **(Medium)** Write a `wrap_tool_call` decorator middleware that prints how long each tool took (`time.perf_counter()`).
4. **(Stretch)** Use `%debug` on `BuggyAudit`: from the `ipdb>` prompt, go `up` until you reach a LangGraph frame. What is the name of the function that called your hook?

<details>
<summary><b>Show solutions</b></summary>

**1.** `B.before_agent, A.before_agent, B.before_model, A.before_model, B.wrap → A.wrap → model, A.after_model, B.after_model …, A.after_agent, B.after_agent`. The first in the list is always the outer layer.

**2.** Each tool round adds 2 messages (AI tool call + `ToolMessage`). One human message + 2 rounds = 5 messages before the third call, so `len(state["messages"]) > 4`.

**3.**
```python
import time
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def timed(request, handler):
    start = time.perf_counter()
    result = handler(request)
    print(f"{request.tool_call['name']}: {(time.perf_counter() - start) * 1000:.1f} ms")
    return result
```

**4.** It varies by version; it is a LangGraph runnable wrapper (look for `invoke` in `langgraph/_internal/_runnable.py`). The point: `up`/`down` let you walk the call stack.
</details>

## ✅ Checkpoint

- [ ] I can name the six hooks and say when each runs.
- [ ] I can predict the order for a stack of middleware (before: in order, after: reversed, wrap: nested).
- [ ] I know what node-style vs wrap-style hooks can do, and when `can_jump_to` is required.
- [ ] I can find the failing hook in a traceback, use `%xmode`, `%debug`, `%pdb`, and keep a failing cell behind a flag.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open the [LangChain v1 release notes](https://docs.langchain.com/oss/python/releases/langchain-v1). Find what replaced `pre_model_hook` and `post_model_hook` of the old `create_react_agent`, and one thing middleware can do that those hooks could not. Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Middleware runs your code at six fixed points of the agent loop without changing the loop.
- Order: `before_*` first-to-last, `after_*` last-to-first, `wrap_*` nested (first = outermost).
- Node-style hooks update state and can `jump_to`; wrap-style hooks control the call itself (skip, retry, override).
- Debug with the `During task with name` line, `%xmode`, `%debug`/`%pdb`, flags for failing cells, and direct hook calls.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 04.02 · Prebuilt middleware tour** → [`04-02-prebuilt-middleware.ipynb`](04-02-prebuilt-middleware.ipynb).

[Module 04 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)